In [1]:
from utils.f_0_dirs import get_data_dirs
dirs = get_data_dirs(segment="calculations")

❌ DATA_DIR path does not exist or is not a directory: /mnt/h/Other computers/My computer/fame_clean/1_FAME_raw_data/2025.07.30


# [read] Create deflator dfs from ONS excel files

### Inflation data
- Import mm23.xlsx from dirs.input_dir, and load sheet 'data'
- The first few rows are filler, so we will need to find the first row where the index is 'CDID' and make that the header
- This is probably row 2, so we should write an assert statement to check that
- Then, filter all columns which are not inflation_cdid = 'L522'
- Rows are either 'filler', 'yearly', 'quarterly' or 'monthly'
- Filter out 'filler' rows. Filter out 'monthly' rows. 'Filter out 'quarterly' rows.
- Filter out 'yearly' rows which are not in the range 2005 - 2026
- Output the remaining series

In [2]:
import pandas as pd

inflation_cdid = 'L522' # CPIH INDEX 0.0
start_year_inf = 2005
xlsx_file = f"{dirs.input_dir}/mm23.xlsx"
xlsx_sheet = "data"

df_raw = pd.read_excel(xlsx_file, sheet_name=xlsx_sheet, engine="calamine")

# Make the first column the index
df_raw.set_index(df_raw.columns[0], inplace=True)

#Make the first row the header
assert df_raw.index[0] == 'CDID', "Expected header not found"
df_raw.columns = df_raw.iloc[0]
df = df_raw[1:]

# Filter all columns where the column name is not our inflation_cdid variable
df = df[df.columns[df.columns == inflation_cdid]]

# Autodetect which index references are 'filler', 'yearly', 'quarterly' or 'monthly', 
# where filler rows don't contain a numeric value within it
filler_rows = df[~df.index.str.contains(r'\d')].index
yearly_rows = df[df.index.str.contains(r'^\d{4}$')].index
quarterly_rows = df[df.index.str.contains(r'^\d{4}Q\d$')].index
monthly_rows = df[df.index.str.contains(r'^\d{4}M\d{2}$')].index

df_yearly_only = df.loc[yearly_rows]
df_yearly_only.index = df_yearly_only.index.astype(int)

# Filter out 'yearly' rows which are not in the range 2005 -
df_year_range = df_yearly_only[(df_yearly_only.index >= start_year_inf)]

# Find the last year in this range which is not a null / NaN
end_year_inf = df_year_range[df_year_range[inflation_cdid].notnull()].index[-1]
df_year_range_filtered = df_year_range[df_year_range.index <= end_year_inf]
# Find where the value is 100
base_year_inf = df_year_range_filtered[df_year_range_filtered[inflation_cdid] == 100].index[0]
if base_year_inf == None:
    raise ValueError("No base year found where the inflation index is 100.")

print(f"Base year: {base_year_inf}, Start year: {start_year_inf}, End year: {end_year_inf}")

df_inflation = df_year_range_filtered[inflation_cdid].astype(float)
print(df_inflation)

Base year: 2015, Start year: 2005, End year: 2025
Title
2005     79.4
2006     81.4
2007     83.3
2008     86.2
2009     87.9
2010     90.1
2011     93.6
2012     96.0
2013     98.2
2014     99.6
2015    100.0
2016    101.0
2017    103.6
2018    106.0
2019    107.8
2020    108.9
2021    111.6
2022    120.5
2023    128.6
2024    132.9
2025    138.0
Name: L522, dtype: float64


### Deflator index

In [ ]:
# Import GDP deflator from input_dir / quarterlynationalaccountsdatatables.xlsx
# Filter out quarters, and then grab the 'YBGB' column, which corresponds to the GDP deflator.
# print resulting dataframe

import pandas as pd

deflator_cdid = 'YBGB' # GDP deflator
start_year_def = 2005

xlsx_file = f"{dirs.input_dir}/quarterlynationalaccountsdatatables.xlsx"
xlsx_sheet = "yearly_variables"
df_raw_def = pd.read_excel(xlsx_file, sheet_name=xlsx_sheet, engine="calamine")

df_raw_def.set_index(df_raw_def.columns[0], inplace=True)
df_raw_def.index = df_raw_def.index.astype(str)
regex_pattern = r'^\d{4}(?:\.0)?$'
yearly_rows_def = df_raw_def[df_raw_def.index.str.contains(regex_pattern, na=False)].index

# (Optional) If it caught the ".0", clean it up so the index is purely the year:
df_yearly_only_def = df_raw_def.loc[yearly_rows_def]
df_yearly_only_def.index = df_yearly_only_def.index.str.replace('.0', '', regex=False).astype(int)

# Parse end year
df_year_range_def = df_yearly_only_def[(df_yearly_only_def.index >= start_year_def)]
end_year_def = df_year_range_def[df_year_range_def[deflator_cdid].notnull()].index[-1]
df_year_range_filtered_def = df_year_range_def[df_year_range_def.index <= end_year_def]

# Parse base year
base_year_def = df_year_range_filtered_def[df_year_range_filtered_def[deflator_cdid] == 100].index[0]
if base_year_def == None:
    raise ValueError("No base year found where the GDP deflator is 100.")

print(f"Base year: {base_year_def}, Start year: {start_year_def}, End year: {end_year_def}")

df_deflator = df_year_range_filtered_def[deflator_cdid].astype(float)

print(df_deflator)

Base year: 2023, Start year: 2005, End year: 2025
Unnamed: 0
2005     64.0
2006     65.9
2007     67.3
2008     69.5
2009     70.9
2010     71.8
2011     73.6
2012     74.7
2013     76.3
2014     77.6
2015     78.1
2016     79.4
2017     80.7
2018     82.2
2019     84.3
2020     88.4
2021     89.0
2022     94.0
2023    100.0
2024    103.9
2025    107.7
Name: YBGB, dtype: float64


In [ ]:
# Combine the two dataframes for columns to be 'cpih' and 'gdpdef'. Index is years
# set a base_year variable to be 2024
# normalise both columns to 2024
import pandas as pd
import numpy as np 

base_year = 2024

df_prices: pd.DataFrame[np.float64] = pd.DataFrame({
    'cpih': df_inflation,
    'gdpdef': df_deflator
})
rebase_dict: dict[str, float] = {
    'cpih': float(df_prices.loc[base_year, 'cpih'] / df_prices.loc[base_year_inf, 'cpih']),
    'gdpdef': float(df_prices.loc[base_year, 'gdpdef'] / df_prices.loc[base_year_def, 'gdpdef'])
}
df_prices_rebased = (df_prices / pd.Series(rebase_dict)).astype(float)

for col in df_prices_rebased.columns:
    val: float = float(df_prices_rebased.loc[base_year, col])
    if round(val, 2) == 100:
        continue
    raise ValueError(f"Expected 100 in {base_year} for {col}, got {val}")

display(df_prices_rebased)

,cpih,gdpdef
2005,59.744169,61.597690
2006,61.249059,63.426372
2007,62.678706,64.773821
2008,64.860798,66.891242
2009,66.139955,68.238691
2010,67.795335,69.104909
2011,70.428894,70.837344
2012,72.234763,71.896054
2013,73.890143,73.435996
2014,74.943567,74.687199


# [read/write] Deflate yearly data

- Load raw_schema from the Excel file in the same way we did before
- raw_schema has a 'deflate' column which is a string, corresponding to every (panel) property in fame_yearly
- For every column in fame_yearly, find the appropriate deflator key from that column
- Note that some variables may have no deflator key (column value is None/Null)
- Match it to the deflator in df_prices_rebased
- And then create a new table called 'fame_yearly_deflated'
- Which is the exact same as fame_yearly, but with the values deflated by the appropriate deflator
- If there was no deflator key, then the value should be copied over exactly

In [5]:
# Load raw_schema from the Excel file in the same way we did before
# raw_schema has a 'deflate' column which is a string, corresponding to every (panel) property in fame_yearly
import pandas as pd
import ibis
from utils.f_0_dirs import get_data_dirs

build_dirs = get_data_dirs(segment="build")
schema_path = build_dirs.input_dir / "raw_properties.xlsx"
schema_source = pd.read_excel(schema_path, sheet_name="raw_properties", engine="calamine",
    dtype={
        "from_raw": str,
        "key": str,
        "type": str,
        "fuzzy_mapping": str,
        "keep": str,
        "in_ln_set": "boolean",
        "may_mix": "boolean",
        "deflate": str,
        "description": str
    }
)
# Iterate over columns where type is boolean and fillna with False
for col in schema_source.select_dtypes(include='boolean').columns:
    schema_source[col] = schema_source[col].fillna(False)
schema_raw: pd.DataFrame = schema_source[schema_source["from_raw"].notna()]
schema_yearly: pd.DataFrame = schema_raw[schema_raw["keep"].isin(["yearly", "all"])]
schema_deflate_mapping = schema_yearly[["key", "deflate"]].set_index("key").to_dict()["deflate"]

print("Deflation mapping:")
for key, deflator in schema_deflate_mapping.items():
    print(f"  {key}: {deflator}")

❌ DATA_DIR path does not exist or is not a directory: /mnt/h/Other computers/My computer/fame_clean/1_FAME_raw_data/2025.07.30
Deflation mapping:
  registered_number: nan
  consolidated: nan
  turnover: gdpdef
  shareholders_funds: gdpdef
  profit_loss_pretax: gdpdef
  employees: nan
  tangibles: gdpdef
  tangibles_land_and_buildings: gdpdef
  tangibles_land_freehold: gdpdef
  tangibles_land_leasehold: gdpdef
  fixed_other: gdpdef
  intangibles: gdpdef
  fixed_total: gdpdef
  liabilities: gdpdef
  total_assets: gdpdef
  liabilites_lt: gdpdef
  cos: gdpdef
  dividends: gdpdef
  r_and_d: gdpdef
  remuneration_employees: cpih
  wages: cpih
  social_security_costs: cpih
  pensions_costs: cpih
  ebitda: gdpdef


- For every column in fame_yearly, find the appropriate deflator key from that column
- Note that some variables may have no deflator key (column value is None/Null)
- Match it to the deflator in df_prices_rebased
- And then create a new table called 'fame_yearly_deflated'
- Which is the exact same as fame_yearly, but with the values deflated by the appropriate deflator
- If there was no deflator key, then the value should be copied over exactly
- Setup complicated sql commands to now get the right deflator column for every column in fame_yearly, and then deflate it by the appropriate deflator

In [6]:
old_table_name = "fame_yearly_filtered"
new_table_name = "fame_yearly_kp"

start_time = pd.Timestamp.now()
con = ibis.duckdb.connect(str(build_dirs.db_path))

# --- 0. Memory Safeguards (Crucial for 8GB RAM limits) ---
# Tell DuckDB it is not allowed to use more than 4GB of RAM for operations. 
# It will automatically spill intermediate calculations to a temporary disk file.
con.raw_sql("PRAGMA memory_limit='4GB'")

table_fame_yearly = con.table(old_table_name)

# --- 1. Push Prices Data into DuckDB ---
if df_prices_rebased.index.name != 'year':
    df_prices_rebased.index.name = 'year'
df_prices_rebased.index = df_prices_rebased.index.astype(int)
df_prices_reset = df_prices_rebased.reset_index()

print("📥 Loading prices data into DuckDB...")
prices_table = con.create_table(new_table_name + "_temp", df_prices_reset, temp=True, overwrite=True)

# --- 2. Build Dynamic Column Selections ---
joined_table = table_fame_yearly.left_join(prices_table, "year")
select_exprs = []

print("⚙️  Building dynamic inflation adjustment abstract syntax tree...")
for col_name in table_fame_yearly.columns:
    deflator = schema_deflate_mapping.get(col_name)
    has_deflator = pd.notna(deflator) and isinstance(deflator, str) and deflator.strip().lower() not in ['nan', 'none', '']
    
    if has_deflator and deflator in prices_table.columns:
        deflated_col = (joined_table[col_name] / (joined_table[deflator] / 100)).name(col_name)
        select_exprs.append(deflated_col)
    else:
        select_exprs.append(joined_table[col_name])

# Build the blueprint of the table (lazy evaluated, no math executed yet)
deflated_table_expr = joined_table.select(*select_exprs)

# --- 3. Create an Empty Target Table ---
print(f"🏗️  Creating empty target table '{new_table_name}'...")
# Grab 0 rows to define the final schema and instantiate it
empty_schema = deflated_table_expr.filter(deflated_table_expr.year == -9999)
con.create_table(new_table_name, empty_schema, overwrite=True)

# --- 4. Process in Memory-Safe Batches (By Year) ---
unique_years = df_prices_reset['year'].unique().tolist()
print(f"🔄 Starting batched deflation across {len(unique_years)} years...")

for y in sorted(unique_years):
    print(f"   [{(pd.Timestamp.now() - start_time).total_seconds():.1f}s] Processing & Materializing Year: {y}")
    
    # Filter the lazy expression to just THIS year
    year_chunk = deflated_table_expr.filter(deflated_table_expr.year == y)
    
    # Execute the chunk and APPEND it to the new table
    con.insert(new_table_name, year_chunk)

# --- 5. Verification ---
print(f"   [{(pd.Timestamp.now() - start_time).total_seconds():.1f}s] Running final checks...")
old_count = table_fame_yearly.count().execute()
new_count = con.table(new_table_name).count().execute()
print(f"✅ Deflation complete! Materialized {new_count:,} rows (matches original: {old_count == new_count}).")

# --- 6. Generate Verification Samples ---
print("🎲 Generating verification sample...")
# Ibis .sample() can be unpredictable across backends, order_by(random) is bulletproof
sample_yearly_df = con.table(old_table_name).order_by(ibis.random()).limit(20).execute()

# Rather than a complex .isin() which 
# triggers full table scans on 14 million rows,
# we turn the 20 sample rows into a temporary in-memory table and inner_join it to pluck them out.
keys_to_fetch = ibis.memtable(sample_yearly_df[["registered_number", "year"]])
sample_deflated_df = (
    con.table(new_table_name)
    .inner_join(keys_to_fetch, ["registered_number", "year"])
    .execute()
)
# Sort this by the same order as the original sample for easy visual comparison
sample_deflated_df = sample_deflated_df.set_index(["registered_number", "year"]).loc[sample_yearly_df.set_index(["registered_number", "year"]).index].reset_index()

display(sample_yearly_df)
display(sample_deflated_df)

📥 Loading prices data into DuckDB...
⚙️  Building dynamic inflation adjustment abstract syntax tree...
🏗️  Creating empty target table 'fame_yearly_kp'...
🔄 Starting batched deflation across 21 years...
   [3.1s] Processing & Materializing Year: 2005
   [3.1s] Processing & Materializing Year: 2006
   [3.6s] Processing & Materializing Year: 2007
   [4.4s] Processing & Materializing Year: 2008
   [4.8s] Processing & Materializing Year: 2009
   [5.6s] Processing & Materializing Year: 2010
   [6.1s] Processing & Materializing Year: 2011
   [6.8s] Processing & Materializing Year: 2012
   [7.3s] Processing & Materializing Year: 2013
   [8.1s] Processing & Materializing Year: 2014
   [8.6s] Processing & Materializing Year: 2015
   [9.2s] Processing & Materializing Year: 2016
   [11.0s] Processing & Materializing Year: 2017
   [11.5s] Processing & Materializing Year: 2018
   [13.0s] Processing & Materializing Year: 2019
   [13.4s] Processing & Materializing Year: 2020
   [13.8s] Processing & M

,registered_number,year,consolidated,turnover,shareholders_funds,profit_loss_pretax,employees,tangibles,tangibles_land_and_buildings,tangibles_land_freehold,...,dividends,depreciation,r_and_d,remuneration_employees,wages,social_security_costs,pensions_costs,other_staff_costs,renumeration_directors,ebitda
0,04053359,2007,True,15057.000,-3321.000,-556.000,180,896.000,NaN,NaN,...,NaN,196.000,NaN,6733.000,5743.000,606.000,384.000,NaN,294.000,-410.000
1,06296993,2018,False,9080.425,3316.398,508.032,110,164.530,NaN,NaN,...,-200.000,109.306,NaN,4623.876,4072.429,445.385,106.062,NaN,629.646,832.770
2,06065335,2021,True,3027.106,45520.403,35283.015,50,52.746,NaN,NaN,...,NaN,45.976,NaN,1798.416,1613.067,150.719,34.630,NaN,38.775,35326.256
3,10040474,2021,False,16607.149,2637.644,1461.960,149,NaN,NaN,NaN,...,NaN,NaN,NaN,1416.830,1247.750,56.406,112.674,NaN,NaN,1461.871
4,01106260,2012,True,331892.000,242399.000,86046.000,2765,100972.000,67116.000,67116.000,...,-25476.000,9518.000,33278.0,122263.000,100468.000,12121.000,9674.000,NaN,3667.000,103014.000
5,06624576,2009,False,167108.000,11253.000,13597.000,1245,12484.000,320.000,74.000,...,NaN,1914.000,1313.0,39118.000,32068.000,2998.000,4052.000,NaN,736.430,16355.000
6,00870766,2013,True,11605.508,8771.881,971.826,23,190.780,0.720,0.720,...,-120.000,48.576,NaN,1505.269,1192.237,155.931,157.101,NaN,661.542,1152.597
7,07088451,2014,True,12444.490,2791.138,-71.830,77,425.708,NaN,NaN,...,NaN,67.147,NaN,1766.611,1636.175,116.340,14.096,NaN,144.162,377.414
8,02058115,2022,False,8530.551,4815.615,704.747,61,5803.072,4593.892,4593.892,...,NaN,400.426,NaN,2020.236,1910.950,35.005,74.281,NaN,195.810,1139.277
9,03584750,2013,False,713.113,320.518,8.166,35,474.115,434.543,393.514,...,NaN,2.120,NaN,571.704,522.621,39.369,9.714,NaN,NaN,10.286


,registered_number,year,consolidated,turnover,shareholders_funds,profit_loss_pretax,employees,tangibles,tangibles_land_and_buildings,tangibles_land_freehold,...,dividends,depreciation,r_and_d,remuneration_employees,wages,social_security_costs,pensions_costs,other_staff_costs,renumeration_directors,ebitda
0,04053359,2007,True,23245.502229,-5127.071322,-858.371471,180,1383.274889,NaN,NaN,...,NaN,196.000,NaN,10742.085234,9162.601441,966.835534,612.648259,NaN,294.000,-632.971768
1,06296993,2018,False,11477.568826,4191.894796,642.147504,110,207.964319,NaN,NaN,...,-252.798054,109.306,NaN,5797.293589,5105.903907,558.411948,132.977734,NaN,629.646,1052.613175
2,06065335,2021,True,3533.891162,53141.234513,41189.946725,50,61.576510,NaN,NaN,...,NaN,45.976,NaN,2141.662065,1920.937315,179.485261,41.239489,NaN,38.775,41240.426948
3,10040474,2021,False,19387.446979,3079.227097,1706.715101,149,NaN,NaN,NaN,...,NaN,NaN,NaN,1687.246478,1485.895833,67.171661,134.178984,NaN,NaN,1706.611201
4,01106260,2012,True,461627.560910,337152.022758,119681.116466,2765,140441.643909,93351.437751,93351.437751,...,-35434.489960,9518.000,46286.267738,169257.840625,139085.387500,16780.009375,13392.443750,NaN,3667.000,143281.855422
5,06624576,2009,False,244887.464034,16490.644570,19925.645980,1245,18294.606488,468.942172,108.442877,...,NaN,1914.000,1924.128350,59144.279863,48485.064846,4532.812287,6126.402730,NaN,736.430,23967.341326
6,00870766,2013,True,15803.568561,11944.933629,1323.364632,23,259.790852,0.980446,0.980446,...,-163.407602,48.576,NaN,2037.171590,1613.526449,211.030854,212.614286,NaN,661.542,1569.525928
7,07088451,2014,True,16662.145760,3737.103585,-96.174446,77,569.987902,NaN,NaN,...,NaN,67.147,NaN,2357.255039,2183.209413,155.236807,18.808819,NaN,144.162,505.326219
8,02058115,2022,False,9428.981371,5322.791473,778.970354,61,6414.246604,5077.716796,5077.716796,...,NaN,400.426,NaN,2228.127505,2107.595477,38.607174,81.924854,NaN,195.810,1259.264684
9,03584750,2013,False,971.067375,436.458980,11.119887,35,645.616625,591.730245,535.859824,...,NaN,2.120,NaN,773.721605,707.294612,53.280449,13.146544,NaN,NaN,14.006755
